In [17]:
# libraries
import glob
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# Question 1
**Please describe the format of the data files. Can you identify any limitations or distortions of the data?**

Structuring the data in separate csv files per state seems annoying to keep up to date. It would be a lot easier just to have one large csv file with all states represented and keep updating it as new data becomes available.

This way, there will be less chances for errors in data entry.

In [18]:
pd.read_csv('datasets/AK.TXT', header=None).head(10)

,0,1,2,3,4
0,AK,F,1910,Mary,14
1,AK,F,1910,Annie,12
2,AK,F,1910,Anna,10
3,AK,F,1910,Margaret,8
4,AK,F,1910,Helen,7
5,AK,F,1910,Elsie,6
6,AK,F,1910,Lucy,6
7,AK,F,1910,Dorothy,5
8,AK,F,1911,Mary,12
9,AK,F,1911,Margaret,7


In [19]:
pd.read_csv('datasets/AL.TXT', header=None).head(10)

,0,1,2,3,4
0,AL,F,1910,Mary,875
1,AL,F,1910,Annie,482
2,AL,F,1910,Willie,257
3,AL,F,1910,Mattie,232
4,AL,F,1910,Ruby,204
5,AL,F,1910,Ethel,197
6,AL,F,1910,Lillie,187
7,AL,F,1910,Ruth,168
8,AL,F,1910,Bessie,162
9,AL,F,1910,Elizabeth,146


In [20]:
pd.read_csv('datasets/WV.TXT', header=None).head(10)

,0,1,2,3,4
0,WV,F,1910,Mary,380
1,WV,F,1910,Virginia,144
2,WV,F,1910,Helen,124
3,WV,F,1910,Ruth,118
4,WV,F,1910,Margaret,115
5,WV,F,1910,Thelma,90
6,WV,F,1910,Gladys,89
7,WV,F,1910,Anna,88
8,WV,F,1910,Ethel,88
9,WV,F,1910,Hazel,88


In [21]:
pd.read_csv('datasets/WY.TXT', header=None).head(10)

,0,1,2,3,4
0,WY,F,1910,Mary,27
1,WY,F,1910,Margaret,22
2,WY,F,1910,Helen,13
3,WY,F,1910,Alice,10
4,WY,F,1910,Dorothy,9
5,WY,F,1910,Frances,8
6,WY,F,1910,Josephine,8
7,WY,F,1910,Lena,8
8,WY,F,1910,Anna,7
9,WY,F,1910,Ruth,7


In [22]:
# get all file names
files = glob.glob('datasets/*.TXT')

# column names
columns = ['state', 'sex', 'year', 'name', 'occurences']

# read files into a list of DataFrames
df_list = [pd.read_csv(file, header=None, names=columns) for file in files]

# combine into df
df = pd.concat(df_list, ignore_index=True)

In [23]:
df.head(10)

,state,sex,year,name,occurences
0,IN,F,1910,Mary,619
1,IN,F,1910,Helen,324
2,IN,F,1910,Ruth,238
3,IN,F,1910,Dorothy,215
4,IN,F,1910,Mildred,200
5,IN,F,1910,Margaret,196
6,IN,F,1910,Thelma,137
7,IN,F,1910,Edna,113
8,IN,F,1910,Martha,112
9,IN,F,1910,Hazel,108


# Question 2
**What is the most popular name of all time? (Of either gender.)**

In [24]:
df.groupby('name')['occurences'].sum().nlargest(1).to_frame()

,occurences
name,
James,5054074


# Question 3
**What is the most gender ambiguous name in 2013? 1945?**

In [25]:
def gender_ambigous_name(year, top_n, n_occurences):
    # print description
    print(f'Top {top_n} names occuring at least {n_occurences} for each sex from {year}:')
    # pivot table
    results_df = df.loc[
        (df['year']==year)
        ].pivot_table(index='name', columns='sex', values='occurences', aggfunc='sum', fill_value=0)
    # filter with n_occurences
    results_df = results_df.loc[
        (results_df['F'] >= n_occurences) &
        (results_df['M'] >= n_occurences)
    ]
    # calculate total occurences
    results_df['Total Occurences'] = results_df['F'] + results_df['M']
    # rename columns
    results_df = results_df.rename(columns={'F':'Female', 'M':'Male'})
    # filter for top_n
    results_df = results_df.nlargest(top_n, ['Total Occurences'])
    return results_df

In [26]:
gender_ambigous_name(year=2013, top_n=10, n_occurences=100)

Top 10 names occuring at least 100 for each sex from 2013:


sex,Female,Male,Total Occurences
name,,,
Jayden,673,14777,15450
Logan,667,12358,13025
Avery,9180,2042,11222
Dylan,571,10125,10696
Ryan,387,9885,10272
Carter,249,9578,9827
Hunter,211,8973,9184
Harper,8284,315,8599
Jordan,1213,7208,8421


In [27]:
gender_ambigous_name(year=1945, top_n=10, n_occurences=100)

Top 10 names occuring at least 100 for each sex from 1945:


sex,Female,Male,Total Occurences
name,,,
James,206,74456,74662
Robert,145,69940,70085
John,144,66117,66261
Mary,59282,145,59427
Carol,30387,143,30530
Jerry,379,14697,15076
Willie,1747,7002,8749
Terry,927,6823,7750
Jean,7423,118,7541


# Question 4
**Of the names represented in the data, find the name that has had the largest percentage increase in popularity since 1980. Largest decrease?**

In [28]:
# Percent Change
q4_p2 = df.loc[
    (df['year'].isin([1980, 2021]))
    ].pivot_table(index='name', columns='year', values='occurences', aggfunc=sum, fill_value=0).reset_index()

q4_p2 = q4_p2.loc[
    (q4_p2[1980] > 0)
]

q4_p2['Percent Increase'] = (((q4_p2[2021] - q4_p2[1980]) / q4_p2[1980]) * 100).round(2)
q4_p2.nlargest(10, 'Percent Increase').set_index('name')

year,1980,2021,Percent Increase
name,,,
Mateo,6,9100,151566.67
Luna,6,8173,136116.67
Aria,5,6348,126860.00
Mila,5,6288,125660.00
Rowan,5,4853,96960.00
Colton,5,4532,90540.00
Skylar,5,3402,67940.00
Cooper,7,4720,67328.57
Walker,5,2886,57620.00


In [29]:
# Percent Change
q4_p2 = df.loc[
    (df['year'].isin([1980, 2021]))
    ].pivot_table(index='name', columns='year', values='occurences', aggfunc=sum, fill_value=0).reset_index()

q4_p2 = q4_p2.loc[
    (q4_p2[2021] > 0)
]

q4_p2['Percent Decrease'] = (((q4_p2[1980] - q4_p2[2021]) / q4_p2[2021]) * 100).round(2)
q4_p2.nlargest(10, 'Percent Decrease').set_index('name')

year,1980,2021,Percent Decrease
name,,,
Jill,4555,5,91000.00
Kristy,4052,5,80940.00
Christy,3672,5,73340.00
Tammy,3131,5,62520.00
Kristin,7732,13,59376.92
Misty,5541,11,50272.73
Kelli,2477,5,49440.00
Mandy,2229,5,44480.00
Brandi,4701,12,39075.00


# Question 5
**Can you identify names that may have had an even larger increase or decrease in popularity?**

In [30]:
# Absolute Difference
q5 = df.loc[
    (df['year'].isin([1980, 2021]))
    ].pivot_table(index='name', columns='year', values='occurences', aggfunc=sum, fill_value=0).reset_index()

q5['Change'] = q5[2021]-q5[1980]
q5.set_index('name')

year,1980,2021,Change
name,,,
Aadhira,0,12,12
Aadhya,0,185,185
Aadi,0,21,21
Aadit,0,5,5
Aadvik,0,22,22
...,...,...,...
Zylo,0,5,5
Zymir,0,96,96
Zyon,0,203,203


In [31]:
q5.nlargest(10, 'Change').set_index('name')

year,1980,2021,Change
name,,,
Liam,71,20272,20201
Noah,899,19114,18215
Olivia,1092,17728,16636
Emma,479,15433,14954
Oliver,335,14616,14281
Ava,62,12759,12697
Charlotte,814,13285,12471
Amelia,742,12952,12210
Elijah,731,12708,11977


In [32]:
q5.nsmallest(10, 'Change').set_index('name')

year,1980,2021,Change
name,,,
Michael,69202,9041,-60161
Jennifer,58529,598,-57931
Jason,48438,2831,-45607
Christopher,49357,5800,-43557
Amanda,35839,619,-35220
David,42169,7843,-34326
Jessica,33992,605,-33387
Melissa,31733,809,-30924
Joshua,36214,5457,-30757


# Bonus
**What insight can you extract from this dataset? Feel free to combine the baby names data with other publicly available datasets or APIs, but be sure to include code for accessing any alternative data that you use.**

**This is an open­ended question and you are free to answer as you see fit. In fact, we would love it if you find an interesting way to look at the data that we haven't thought of!**

**Please provide us with both your code and an informative write­up of your results. The code should be in a runnable form. Do not assume that we have a copy of the data set or that we are familiar with the build procedures for your chosen language.**